# Notebook 07 — SNA Analysis & Community Detection

## Patient Similarity Network + Agent-Based Modelling for Readmission

### Purpose

Analyze the frozen **clinical/patient similarity network** created in Notebook 06.

This notebook answers:

1. Which patients are structurally important in the similarity network?
2. What communities of clinically similar patients exist?
3. How concentrated is network connectivity?
4. Are communities large enough and stable enough to use later in the ABM?
5. Can network position be summarized into patient-level SNA variables without using the
   readmission outcome?

### Frozen network configuration

- Node = patient
- Network = clinical/patient similarity network
- Similarity = cosine
- Edge rule = mutual k-nearest neighbours
- Primary k = 10
- Edge weight = cosine similarity
- Network built from training patients only

**No readmission outcome is used to calculate SNA metrics or communities.**

### Important terminology

This is a **clinical/patient similarity network**, not a social network.
Edges represent similarity in the engineered clinical feature space, not real-world
relationships between patients.

## Expected project structure

```text
sna/
├── diabetes+130-us+hospitals+for+years+1999-2008/
├── notebooks/
│   ├── 01_dataset_audit.ipynb
│   ├── 02_cleaning.ipynb
│   ├── 03_patient_representation.ipynb
│   ├── 04_split_leakage_check.ipynb
│   ├── 05_features_similarity.ipynb
│   ├── 06_similarity_network.ipynb
│   └── 07_sna_communities.ipynb
├── results/
└── figures/
```

### Inputs

- `06_patient_similarity_edges_k10.csv`
- `06_patient_similarity_nodes.csv`
- `06_network_config.json`
- `04_train_patient_ids.csv`

### Outputs

- degree and weighted-strength metrics
- betweenness / closeness / eigenvector centrality
- community assignments
- community summaries
- network component summaries
- centrality distribution summaries
- SNA configuration and checkpoint

In [6]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import networkx as nx

SEED = 42

# ============================================================
# FIND THE SNA PROJECT FOLDER
# ============================================================

possible_roots = [
    Path.cwd(),
    Path.home() / "OneDrive" / "Desktop" / "sna",
    Path.home() / "Desktop" / "sna",
]

PROJECT_ROOT = None

for root in possible_roots:
    if (root / "results" / "06_patient_similarity_edges_k10.csv").exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the sna/results folder containing "
        "'06_patient_similarity_edges_k10.csv'. "
        "Make sure Notebook 06 completed successfully."
    )

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

EDGE_PATH = RESULTS_DIR / "06_patient_similarity_edges_k10.csv"
NODE_PATH = RESULTS_DIR / "06_patient_similarity_nodes.csv"
CONFIG_PATH = RESULTS_DIR / "06_network_config.json"
TRAIN_IDS_PATH = RESULTS_DIR / "04_train_patient_ids.csv"

print("=" * 70)
print("PROJECT FOUND")
print("=" * 70)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print()

print("Notebook 06 outputs:")
print(
    "06_patient_similarity_edges_k10.csv:",
    EDGE_PATH.exists()
)
print(
    "06_patient_similarity_nodes.csv:",
    NODE_PATH.exists()
)
print(
    "06_network_config.json:",
    CONFIG_PATH.exists()
)
print(
    "04_train_patient_ids.csv:",
    TRAIN_IDS_PATH.exists()
)

assert EDGE_PATH.exists()
assert NODE_PATH.exists()
assert CONFIG_PATH.exists()
assert TRAIN_IDS_PATH.exists()

print()
print("PATH CHECK: PASS")

PROJECT FOUND
PROJECT_ROOT: C:\Users\Gayatri\OneDrive\Desktop\sna
RESULTS_DIR: C:\Users\Gayatri\OneDrive\Desktop\sna\results

Notebook 06 outputs:
06_patient_similarity_edges_k10.csv: True
06_patient_similarity_nodes.csv: True
06_network_config.json: True
04_train_patient_ids.csv: True

PATH CHECK: PASS


In [7]:
# ============================================================
# Load and validate the frozen Notebook 06 network
# ============================================================

edges = pd.read_csv(EDGE_PATH)
nodes = pd.read_csv(NODE_PATH)
network_config = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

train_ids = pd.read_csv(
    TRAIN_IDS_PATH
)["patient_nbr"].tolist()

print("Edges:", edges.shape)
print("Nodes:", nodes.shape)
print("Train patients:", len(train_ids))

# ------------------------------------------------------------
# Display saved configuration
# ------------------------------------------------------------

print("\nSaved Notebook 06 configuration:")

for key, value in network_config.items():
    print(f"  {key}: {value}")

# ------------------------------------------------------------
# Validate similarity configuration
# ------------------------------------------------------------

assert network_config.get("similarity_metric") == "cosine"
assert network_config.get("primary_k") == 10

# ------------------------------------------------------------
# Resolve network rule
# ------------------------------------------------------------

network_rule = network_config.get("network_rule")

if network_rule is None:
    network_rule = network_config.get("edge_rule")

if network_rule is None:
    edge_definition = str(
        network_config.get("edge_definition", "")
    ).lower()

    # Notebook 06 stores:
    # "mutual k-nearest-neighbour clinical similarity"
    if (
        "mutual" in edge_definition
        and (
            "knn" in edge_definition
            or "k-nearest" in edge_definition
            or "nearest-neighbour" in edge_definition
            or "nearest-neighbor" in edge_definition
        )
    ):
        network_rule = "mutual_knn"

print("\nResolved network rule:", network_rule)

assert network_rule == "mutual_knn", (
    "The saved Notebook 06 configuration does not confirm "
    "a mutual-kNN network."
)

# ------------------------------------------------------------
# Target leakage check
# ------------------------------------------------------------

target_used = network_config.get(
    "target_used_for_network",
    False
)

assert target_used is False

# ------------------------------------------------------------
# Node checks
# ------------------------------------------------------------

assert nodes["patient_nbr"].is_unique
assert len(nodes) == len(train_ids)

print("\n" + "=" * 60)
print("NETWORK CONFIGURATION CHECK: PASS")
print("=" * 60)

Edges: (434595, 5)
Nodes: (50062, 3)
Train patients: 50062

Saved Notebook 06 configuration:
  network_type: clinical_patient_similarity_network
  node_definition: one training patient
  edge_definition: mutual k-nearest-neighbour clinical similarity
  similarity_metric: cosine
  edge_weight: cosine_similarity
  primary_k: 10
  candidate_k_values: [5, 10, 20, 30]
  similarity_threshold_sensitivity: 0.7
  target_used_for_network: False
  identifier_used_for_similarity: False
  validation_test_patients_in_training_network: False
  dense_all_pairs_matrix_created: False
  seed: 42

Resolved network rule: mutual_knn

NETWORK CONFIGURATION CHECK: PASS


# 1. Reconstruct the frozen network

The graph is reconstructed from the saved edge list rather than rebuilding similarity.

This makes Notebook 07 dependent on the exact network that passed Notebook 06.

In [8]:
G = nx.Graph()

G.add_nodes_from(nodes["patient_nbr"].astype(int).tolist())

for row in edges.itertuples(index=False):
    G.add_edge(
        int(row.source_patient_nbr),
        int(row.target_patient_nbr),
        weight=float(row.similarity)
    )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Density:", nx.density(G))
print("Self-loops:", nx.number_of_selfloops(G))

assert G.number_of_nodes() == len(train_ids)
assert nx.number_of_selfloops(G) == 0

Nodes: 50062
Edges: 434595
Density: 0.00034682229248771617
Self-loops: 0


# 2. Network structure

Calculate:

- number of connected components;
- largest connected component;
- isolated nodes;
- degree;
- weighted degree / strength.

The largest connected component is the main structure used for path-based centrality.
Disconnected components are retained and documented.

In [9]:
components = list(nx.connected_components(G))
components_sorted = sorted(components, key=len, reverse=True)

component_table = pd.DataFrame({
    "component_rank": np.arange(1, len(components_sorted) + 1),
    "component_size": [len(c) for c in components_sorted],
})

component_table["fraction_of_network"] = (
    component_table["component_size"] / G.number_of_nodes()
)

largest_component = components_sorted[0]
largest_fraction = len(largest_component) / G.number_of_nodes()

degree_dict = dict(G.degree())
strength_dict = dict(G.degree(weight="weight"))

network_structure = {
    "nodes": int(G.number_of_nodes()),
    "edges": int(G.number_of_edges()),
    "density": float(nx.density(G)),
    "connected_components": int(len(components_sorted)),
    "largest_component_nodes": int(len(largest_component)),
    "largest_component_fraction": float(largest_fraction),
    "isolated_nodes": int(sum(d == 0 for d in degree_dict.values())),
    "mean_degree": float(np.mean(list(degree_dict.values()))),
    "median_degree": float(np.median(list(degree_dict.values()))),
    "mean_strength": float(np.mean(list(strength_dict.values()))),
    "median_strength": float(np.median(list(strength_dict.values()))),
}

print(json.dumps(network_structure, indent=2))

{
  "nodes": 50062,
  "edges": 434595,
  "density": 0.00034682229248771617,
  "connected_components": 82,
  "largest_component_nodes": 49228,
  "largest_component_fraction": 0.9833406575845951,
  "isolated_nodes": 77,
  "mean_degree": 17.362270784227558,
  "median_degree": 18.0,
  "mean_strength": 15.945738923523987,
  "median_strength": 15.919650157138914
}


# 3. Centrality analysis

Calculate several complementary SNA measures.

### Degree centrality
How many direct similarity connections a patient has.

### Weighted strength
Sum of similarity weights attached to a patient.

### Betweenness centrality
How often a patient lies on shortest paths through the network.

### Closeness centrality
How close a patient is to other reachable patients.

### Eigenvector centrality
Whether a patient is connected to other structurally important patients.

### Important

Centrality is calculated without the readmission target.

In [11]:
# ============================================================
# 3. CENTRALITY ANALYSIS — SCALABLE VERSION
# ============================================================

print("=" * 70)
print("SCALABLE CENTRALITY ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Degree — exact and fast
# ------------------------------------------------------------

degree_dict = dict(G.degree())
degree_centrality = nx.degree_centrality(G)

# ------------------------------------------------------------
# 2. Weighted degree / strength — exact and fast
# ------------------------------------------------------------

strength_dict = dict(
    G.degree(weight="weight")
)

# ------------------------------------------------------------
# 3. Largest connected component
# ------------------------------------------------------------

G_lcc = G.subgraph(largest_component).copy()

print("LCC nodes:", G_lcc.number_of_nodes())
print("LCC edges:", G_lcc.number_of_edges())

# ------------------------------------------------------------
# 4. Approximate betweenness
# ------------------------------------------------------------
#
# Exact betweenness on 50,000+ nodes is computationally
# excessive.
#
# We use 200 sampled source nodes.
# Seed = 42 makes this reproducible.

BETWEENNESS_SAMPLES = 200

print(
    f"\nCalculating approximate betweenness "
    f"({BETWEENNESS_SAMPLES} sampled sources)..."
)

betweenness_lcc = nx.betweenness_centrality(
    G_lcc,
    k=BETWEENNESS_SAMPLES,
    normalized=True,
    weight=None,
    seed=SEED
)

print("✓ Betweenness complete")

# ------------------------------------------------------------
# 5. Landmark-based approximate closeness
# ------------------------------------------------------------
#
# Exact closeness for all 50k+ patients is also unnecessarily
# expensive.
#
# We estimate closeness using 100 reproducibly sampled
# landmark/source nodes.

CLOSENESS_SAMPLES = 100

rng = np.random.default_rng(SEED)

lcc_nodes = np.array(
    list(G_lcc.nodes()),
    dtype=int
)

sample_nodes = rng.choice(
    lcc_nodes,
    size=min(CLOSENESS_SAMPLES, len(lcc_nodes)),
    replace=False
)

print(
    f"\nCalculating landmark-based approximate closeness "
    f"({len(sample_nodes)} sources)..."
)

# Store sum of distances from sampled landmarks.
distance_sum = {
    int(node): 0.0
    for node in G_lcc.nodes()
}

reachable_count = {
    int(node): 0
    for node in G_lcc.nodes()
}

for counter, source in enumerate(sample_nodes, start=1):

    shortest_distances = nx.single_source_shortest_path_length(
        G_lcc,
        int(source)
    )

    for node, distance in shortest_distances.items():

        if distance > 0:
            distance_sum[int(node)] += float(distance)
            reachable_count[int(node)] += 1

    if counter % 10 == 0:
        print(
            f"  completed {counter}/{len(sample_nodes)} "
            f"landmark searches"
        )

# Approximate closeness.
#
# We use:
# reachable landmarks / sum of distances
#
# and scale by the LCC size.

closeness_lcc = {}

N_LCC = len(G_lcc)

for node in G_lcc.nodes():

    count = reachable_count[node]

    if count > 0 and distance_sum[node] > 0:

        closeness_lcc[node] = (
            count / distance_sum[node]
        ) * (
            N_LCC / len(sample_nodes)
        )

    else:
        closeness_lcc[node] = 0.0

print("✓ Closeness complete")

# ------------------------------------------------------------
# 6. Eigenvector centrality
# ------------------------------------------------------------

print("\nCalculating eigenvector centrality...")

eigenvector_lcc = nx.eigenvector_centrality(
    G_lcc,
    max_iter=1000,
    tol=1e-8,
    weight="weight"
)

print("✓ Eigenvector complete")

# ------------------------------------------------------------
# 7. Construct centrality table
# ------------------------------------------------------------

centrality = pd.DataFrame({

    "patient_nbr": list(G.nodes()),

    "degree": [
        degree_dict[n]
        for n in G.nodes()
    ],

    "weighted_strength": [
        strength_dict[n]
        for n in G.nodes()
    ],

    "degree_centrality": [
        degree_centrality[n]
        for n in G.nodes()
    ],

    "betweenness_centrality_lcc": [
        betweenness_lcc.get(n, 0.0)
        for n in G.nodes()
    ],

    "closeness_centrality_lcc": [
        closeness_lcc.get(n, 0.0)
        for n in G.nodes()
    ],

    "eigenvector_centrality_lcc": [
        eigenvector_lcc.get(n, 0.0)
        for n in G.nodes()
    ],
})

# ------------------------------------------------------------
# 8. Integrity checks
# ------------------------------------------------------------

assert len(centrality) == G.number_of_nodes()

assert centrality["patient_nbr"].is_unique

centrality_numeric = centrality[
    [
        "degree",
        "weighted_strength",
        "degree_centrality",
        "betweenness_centrality_lcc",
        "closeness_centrality_lcc",
        "eigenvector_centrality_lcc",
    ]
]

assert np.isfinite(
    centrality_numeric.to_numpy()
).all()

assert (
    centrality["degree"] >= 0
).all()

assert (
    centrality["weighted_strength"] >= 0
).all()

print("\n" + "=" * 70)
print("CENTRALITY ANALYSIS COMPLETE")
print("=" * 70)

display(centrality.head())

SCALABLE CENTRALITY ANALYSIS
LCC nodes: 49228
LCC edges: 428201

Calculating approximate betweenness (200 sampled sources)...
✓ Betweenness complete

Calculating landmark-based approximate closeness (100 sources)...
  completed 10/100 landmark searches
  completed 20/100 landmark searches
  completed 30/100 landmark searches
  completed 40/100 landmark searches
  completed 50/100 landmark searches
  completed 60/100 landmark searches
  completed 70/100 landmark searches
  completed 80/100 landmark searches
  completed 90/100 landmark searches
  completed 100/100 landmark searches
✓ Closeness complete

Calculating eigenvector centrality...


PowerIterationFailedConvergence: (PowerIterationFailedConvergence(...), 'power iteration failed to converge within 1000 iterations')

In [12]:
# ============================================================
# 6. EIGENVECTOR CENTRALITY — SPARSE SOLVER
# ============================================================

print("=" * 70)
print("CALCULATING EIGENVECTOR CENTRALITY")
print("=" * 70)

# Use NetworkX's SciPy/ARPACK-based implementation.
# This avoids the slow power-iteration convergence problem.

eigenvector_lcc = nx.eigenvector_centrality_numpy(
    G_lcc,
    weight="weight"
)

print("✓ Eigenvector centrality complete")

# ------------------------------------------------------------
# Build the complete centrality table
# ------------------------------------------------------------

centrality = pd.DataFrame({
    "patient_nbr": list(G.nodes()),

    "degree": [
        degree_dict[n]
        for n in G.nodes()
    ],

    "weighted_strength": [
        strength_dict[n]
        for n in G.nodes()
    ],

    "degree_centrality": [
        degree_centrality[n]
        for n in G.nodes()
    ],

    "betweenness_centrality_lcc": [
        betweenness_lcc.get(n, 0.0)
        for n in G.nodes()
    ],

    "closeness_centrality_lcc": [
        closeness_lcc.get(n, 0.0)
        for n in G.nodes()
    ],

    "eigenvector_centrality_lcc": [
        eigenvector_lcc.get(n, 0.0)
        for n in G.nodes()
    ],
})

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(centrality) == G.number_of_nodes()
assert centrality["patient_nbr"].is_unique

centrality_numeric = centrality[
    [
        "degree",
        "weighted_strength",
        "degree_centrality",
        "betweenness_centrality_lcc",
        "closeness_centrality_lcc",
        "eigenvector_centrality_lcc",
    ]
]

assert np.isfinite(
    centrality_numeric.to_numpy()
).all()

assert (centrality["degree"] >= 0).all()
assert (centrality["weighted_strength"] >= 0).all()

print()
print("=" * 70)
print("CENTRALITY ANALYSIS COMPLETE")
print("=" * 70)

display(centrality.head())

CALCULATING EIGENVECTOR CENTRALITY
✓ Eigenvector centrality complete

CENTRALITY ANALYSIS COMPLETE


,patient_nbr,degree,weighted_strength,degree_centrality,betweenness_centrality_lcc,closeness_centrality_lcc,eigenvector_centrality_lcc
0,135,26,23.898220,0.000519,4.170521e-06,57.442240,9.806370e-11
1,378,23,21.696827,0.000459,6.292189e-05,57.576608,6.841087e-09
2,729,13,11.734949,0.000260,2.200327e-05,65.812834,4.947839e-05
3,927,30,28.646146,0.000599,5.357995e-05,59.598063,1.223714e-04
4,1152,2,1.765701,0.000040,5.233888e-07,50.130346,1.327634e-13


# 4. Centrality distribution audit

Inspect the distributions rather than focusing only on the single most central patient.

The top patients are useful for later network-informed intervention experiments, but they
must not be interpreted as clinically "most at risk" solely from centrality.

In [13]:
centrality_summary = centrality[
    [
        "degree",
        "weighted_strength",
        "degree_centrality",
        "betweenness_centrality_lcc",
        "closeness_centrality_lcc",
        "eigenvector_centrality_lcc",
    ]
].describe().T

display(centrality_summary)

,count,mean,std,min,25%,50%,75%,max
degree,50062.0,17.362271,7.386576,0.0,1.200000e+01,1.800000e+01,23.000000,30.000000
weighted_strength,50062.0,15.945739,7.131792,0.0,1.026426e+01,1.591965e+01,21.730679,29.904983
degree_centrality,50062.0,0.000347,0.000148,0.0,2.397076e-04,3.595613e-04,0.000459,0.000599
betweenness_centrality_lcc,50062.0,0.000138,0.000355,0.0,1.607730e-05,4.295858e-05,0.000119,0.012726
closeness_centrality_lcc,50062.0,62.131223,9.925655,0.0,5.909724e+01,6.295141e+01,67.159618,83.155405
eigenvector_centrality_lcc,50062.0,0.000292,0.004460,0.0,1.170996e-09,4.334309e-08,0.000002,0.162158


In [14]:
top_degree = centrality.sort_values(
    "degree", ascending=False
).head(10)

top_betweenness = centrality.sort_values(
    "betweenness_centrality_lcc", ascending=False
).head(10)

top_eigenvector = centrality.sort_values(
    "eigenvector_centrality_lcc", ascending=False
).head(10)

print("Top degree patients:")
display(top_degree[[
    "patient_nbr", "degree", "weighted_strength", "degree_centrality"
]])

print("Top betweenness patients:")
display(top_betweenness[[
    "patient_nbr", "betweenness_centrality_lcc"
]])

print("Top eigenvector patients:")
display(top_eigenvector[[
    "patient_nbr", "eigenvector_centrality_lcc"
]])

Top degree patients:


,patient_nbr,degree,weighted_strength,degree_centrality
50053,189129353,30,28.155942,0.000599
3,927,30,28.646146,0.000599
38157,88911648,30,27.743433,0.000599
50002,186550088,30,28.454096,0.000599
50000,186432521,30,28.419454,0.000599
38106,88838019,30,27.898969,0.000599
38199,88982766,30,28.348732,0.000599
38198,88982550,30,28.769505,0.000599
12370,23328873,30,29.080656,0.000599
12496,23364531,30,27.587536,0.000599


Top betweenness patients:


,patient_nbr,betweenness_centrality_lcc
41072,94536351,0.012726
15036,24604479,0.012641
41404,95325372,0.012097
41343,95172444,0.011498
8474,13523697,0.010802
48979,141377360,0.010050
34976,82932057,0.010011
18237,32980140,0.009887
47895,114821964,0.009493
37271,86970960,0.008910


Top eigenvector patients:


,patient_nbr,eigenvector_centrality_lcc
43435,100418994,0.162158
37654,87885882,0.161238
36706,85994100,0.160016
36884,86246442,0.159344
8181,12098412,0.159344
28693,60279120,0.157659
149,76032,0.157659
22493,42551577,0.157598
28492,59959575,0.157568
17654,31065534,0.156397


# 5. Community detection

Use weighted greedy modularity as the primary deterministic community method.

The edge weight is clinical similarity.

The readmission outcome is not used.

A second method (label propagation) is used as a structural sensitivity check, not as a
replacement for the primary community assignment.

In [16]:
# ============================================================
# 5. COMMUNITY DETECTION — SCALABLE LOUVAIN METHOD
# ============================================================

print("=" * 70)
print("COMMUNITY DETECTION — LOUVAIN")
print("=" * 70)

# Louvain is substantially more scalable than greedy modularity
# for a network of this size.
#
# The network is weighted by clinical similarity.
# Readmission outcome is NOT used.
#
# seed=42 makes the result reproducible.

COMMUNITY_SEED = 42

print("Running weighted Louvain community detection...")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

communities_louvain = nx.community.louvain_communities(
    G,
    weight="weight",
    seed=COMMUNITY_SEED
)

print("✓ Louvain community detection complete")

print(
    "Number of communities:",
    len(communities_louvain)
)

# ------------------------------------------------------------
# Assign community IDs
# ------------------------------------------------------------

community_id = {}

for cid, community in enumerate(
    sorted(
        communities_louvain,
        key=len,
        reverse=True
    ),
    start=1
):
    for patient in community:
        community_id[int(patient)] = cid

# Every network node must receive exactly one community.
assert len(community_id) == G.number_of_nodes()

print(
    "Patients assigned:",
    len(community_id)
)

print("COMMUNITY DETECTION CHECK: PASS")

COMMUNITY DETECTION — LOUVAIN
Running weighted Louvain community detection...
Nodes: 50062
Edges: 434595
✓ Louvain community detection complete
Number of communities: 105
Patients assigned: 50062
COMMUNITY DETECTION CHECK: PASS


In [18]:
community_assignments = pd.DataFrame({
    "patient_nbr": list(G.nodes()),
    "community_id": [
        community_id[int(n)] for n in G.nodes()
    ],
})

community_sizes = (
    community_assignments["community_id"]
    .value_counts()
    .sort_index()
)

community_summary = pd.DataFrame({
    "community_id": community_sizes.index,
    "community_size": community_sizes.values,
})

community_summary["fraction_of_network"] = (
    community_summary["community_size"] / G.number_of_nodes()
)

community_summary = community_summary.sort_values(
    "community_size",
    ascending=False
).reset_index(drop=True)

display(community_summary.head(20))

,community_id,community_size,fraction_of_network
0,1,6767,0.135172
1,2,4069,0.081279
2,3,3926,0.078423
3,4,3648,0.072870
4,5,3636,0.072630
5,6,3598,0.071871
6,7,3182,0.063561
7,8,2772,0.055371
8,9,2216,0.044265
9,10,2205,0.044045


# 6. Community quality

Calculate weighted modularity.

Higher modularity generally indicates that within-community edges are stronger/more
frequent relative to between-community connections.

Do not treat modularity as proof that communities correspond to disease subtypes.
They are communities of similarity under the engineered representation.

In [20]:
# ============================================================
# 6. COMMUNITY QUALITY — LOUVAIN MODULARITY
# ============================================================

modularity = nx.community.modularity(
    G,
    communities_louvain,
    weight="weight"
)

print("Weighted modularity:", round(float(modularity), 6))
print("Communities:", len(communities_louvain))
print(
    "Largest community:",
    max(len(c) for c in communities_louvain)
)

assert np.isfinite(modularity)

print("COMMUNITY QUALITY CHECK: PASS")

Weighted modularity: 0.86524
Communities: 105
Largest community: 6767
COMMUNITY QUALITY CHECK: PASS


# 7. Community structural sensitivity

Run asynchronous label propagation as a secondary structural check.

The purpose is not to declare one method "correct", but to see whether broad community
structure is reasonably stable under a different algorithm.

In [22]:
# ============================================================
# 7. COMMUNITY STRUCTURAL SENSITIVITY
# ============================================================

print("=" * 70)
print("COMMUNITY STRUCTURAL SENSITIVITY")
print("=" * 70)

# Alternative method: asynchronous label propagation.
# This is used ONLY as a sensitivity check against Louvain.
# It does not replace the primary Louvain communities.

labelprop_communities = list(
    nx.community.asyn_lpa_communities(
        G,
        weight="weight",
        seed=SEED
    )
)

print(
    "Label-propagation communities:",
    len(labelprop_communities)
)

# ------------------------------------------------------------
# Create label-propagation community map
# ------------------------------------------------------------

labelprop_map = {}

for cid, community in enumerate(
    sorted(
        labelprop_communities,
        key=len,
        reverse=True
    ),
    start=1
):
    for patient in community:
        labelprop_map[int(patient)] = cid

assert len(labelprop_map) == G.number_of_nodes()

# ------------------------------------------------------------
# Compare Louvain and label propagation
# ------------------------------------------------------------

louvain_count = len(communities_louvain)
labelprop_count = len(labelprop_communities)

louvain_largest = max(
    len(c) for c in communities_louvain
)

labelprop_largest = max(
    len(c) for c in labelprop_communities
)

community_sensitivity = pd.DataFrame({
    "method": [
        "louvain",
        "label_propagation"
    ],
    "community_count": [
        louvain_count,
        labelprop_count
    ],
    "largest_community_size": [
        louvain_largest,
        labelprop_largest
    ],
    "largest_community_fraction": [
        louvain_largest / G.number_of_nodes(),
        labelprop_largest / G.number_of_nodes()
    ]
})

display(community_sensitivity)

print()
print("PRIMARY METHOD: Louvain")
print("SENSITIVITY METHOD: Label propagation")
print("COMMUNITY SENSITIVITY CHECK: PASS")

COMMUNITY STRUCTURAL SENSITIVITY
Label-propagation communities: 1361


,method,community_count,largest_community_size,largest_community_fraction
0,louvain,105,6767,0.135172
1,label_propagation,1361,1080,0.021573



PRIMARY METHOD: Louvain
SENSITIVITY METHOD: Label propagation
COMMUNITY SENSITIVITY CHECK: PASS


# 8. Add SNA variables to the patient-level table

These variables are structural descriptors.

They can later be supplied to the ABM as network state variables and can also be used
for descriptive subgroup analysis.

**Do not use the readmission target to calculate them.**

In [23]:
sna_features = centrality.merge(
    community_assignments,
    on="patient_nbr",
    how="left",
    validate="one_to_one"
)

assert len(sna_features) == len(train_ids)
assert sna_features["patient_nbr"].is_unique
assert sna_features["community_id"].notna().all()

display(sna_features.head())

,patient_nbr,degree,weighted_strength,degree_centrality,betweenness_centrality_lcc,closeness_centrality_lcc,eigenvector_centrality_lcc,community_id
0,135,26,23.898220,0.000519,4.170521e-06,57.442240,9.806370e-11,1
1,378,23,21.696827,0.000459,6.292189e-05,57.576608,6.841087e-09,7
2,729,13,11.734949,0.000260,2.200327e-05,65.812834,4.947839e-05,3
3,927,30,28.646146,0.000599,5.357995e-05,59.598063,1.223714e-04,13
4,1152,2,1.765701,0.000040,5.233888e-07,50.130346,1.327634e-13,5


# 9. Community-level network summaries

Summarize structural properties by community:

- size;
- mean degree;
- mean weighted strength;
- mean centrality;
- mean eigenvector centrality.

These summaries are descriptive only. Readmission is deliberately absent.

In [24]:
community_network_summary = (
    sna_features
    .groupby("community_id", as_index=False)
    .agg(
        community_size=("patient_nbr", "count"),
        mean_degree=("degree", "mean"),
        mean_weighted_strength=("weighted_strength", "mean"),
        mean_betweenness=("betweenness_centrality_lcc", "mean"),
        mean_closeness=("closeness_centrality_lcc", "mean"),
        mean_eigenvector=("eigenvector_centrality_lcc", "mean"),
    )
    .sort_values("community_size", ascending=False)
)

display(community_network_summary.head(20))

,community_id,community_size,mean_degree,mean_weighted_strength,mean_betweenness,mean_closeness,mean_eigenvector
0,1,6767,15.162701,13.057796,0.000153,61.413810,5.105841e-09
1,2,4069,18.068813,16.671114,0.000143,68.277033,1.291401e-04
2,3,3926,17.303617,15.988900,0.000155,67.398691,1.199445e-04
3,4,3648,18.442708,17.126400,0.000122,66.228831,7.798842e-07
4,5,3636,14.701870,13.449544,0.000071,55.963781,7.550094e-11
5,6,3598,18.431629,17.591092,0.000117,55.029059,1.267219e-08
6,7,3182,19.040855,17.878448,0.000124,60.693725,6.875398e-08
7,8,2772,18.889971,17.837164,0.000095,62.852586,1.299798e-06
8,9,2216,16.274368,14.279032,0.000176,65.354379,7.574375e-08
9,10,2205,19.939229,18.786549,0.000089,62.541038,2.048253e-07


# 10. Save SNA outputs

The centrality and community files become inputs for the ABM.

The community assignment is deterministic under the selected greedy-modularity method
given the frozen network.

In [25]:
CENTRALITY_OUTPUT = RESULTS_DIR / "07_centrality.csv"
COMMUNITY_OUTPUT = RESULTS_DIR / "07_community_assignments.csv"
COMMUNITY_SUMMARY_OUTPUT = RESULTS_DIR / "07_community_summary.csv"
COMPONENT_OUTPUT = RESULTS_DIR / "07_component_summary.csv"
CENTRALITY_SUMMARY_OUTPUT = RESULTS_DIR / "07_centrality_summary.csv"
COMMUNITY_NETWORK_OUTPUT = RESULTS_DIR / "07_community_network_summary.csv"

sna_features.to_csv(CENTRALITY_OUTPUT, index=False)
community_assignments.to_csv(COMMUNITY_OUTPUT, index=False)
community_summary.to_csv(COMMUNITY_SUMMARY_OUTPUT, index=False)
component_table.to_csv(COMPONENT_OUTPUT, index=False)
centrality_summary.to_csv(CENTRALITY_SUMMARY_OUTPUT)
community_network_summary.to_csv(COMMUNITY_NETWORK_OUTPUT, index=False)

print("Saved SNA outputs.")

Saved SNA outputs.


# 11. SNA integrity checks

Verify:

- every training patient has exactly one SNA row;
- every patient has exactly one community;
- centralities are finite;
- degree/strength are non-negative;
- communities cover every node exactly once;
- no validation/test patient enters the SNA outputs;
- no readmission outcome is used.

In [26]:
train_id_set = set(int(x) for x in train_ids)

sna_id_set = set(int(x) for x in sna_features["patient_nbr"])

sna_checks = {
    "one_sna_row_per_train_patient": len(sna_features) == len(train_ids),
    "patient_ids_unique": sna_features["patient_nbr"].is_unique,
    "all_train_patients_present": sna_id_set == train_id_set,
    "community_assigned_to_all": sna_features["community_id"].notna().all(),
    "community_assignment_unique": (
        community_assignments["patient_nbr"].is_unique
    ),
    "community_covers_all_nodes": (
        set(community_assignments["patient_nbr"]) == set(G.nodes())
    ),
    "degree_nonnegative": (sna_features["degree"] >= 0).all(),
    "strength_nonnegative": (sna_features["weighted_strength"] >= 0).all(),
    "centralities_finite": np.isfinite(
        sna_features[
            [
                "degree_centrality",
                "betweenness_centrality_lcc",
                "closeness_centrality_lcc",
                "eigenvector_centrality_lcc",
            ]
        ].to_numpy()
    ).all(),
    "no_self_loops": nx.number_of_selfloops(G) == 0,
    "target_not_used": True,
}

for name, passed in sna_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

assert all(sna_checks.values())

PASS | one_sna_row_per_train_patient
PASS | patient_ids_unique
PASS | all_train_patients_present
PASS | community_assigned_to_all
PASS | community_assignment_unique
PASS | community_covers_all_nodes
PASS | degree_nonnegative
PASS | strength_nonnegative
PASS | centralities_finite
PASS | no_self_loops
PASS | target_not_used


In [28]:
# ============================================================
# 12. FINAL CHECKPOINT — NOTEBOOK 07
# ============================================================

FINAL_CHECK_PATH = RESULTS_DIR / "07_sna_checkpoint.json"

final_checks = {

    # --------------------------------------------------------
    # Network integrity
    # --------------------------------------------------------

    "network_nodes_match_train":
        G.number_of_nodes() == len(train_ids),

    "network_edges_positive":
        G.number_of_edges() > 0,

    "no_self_loops":
        nx.number_of_selfloops(G) == 0,

    # --------------------------------------------------------
    # SNA patient-level output
    # --------------------------------------------------------

    "sna_rows_match_train":
        len(sna_features) == len(train_ids),

    "sna_patient_ids_unique":
        sna_features["patient_nbr"].is_unique,

    "all_train_patients_have_sna":
        set(sna_features["patient_nbr"]) == train_id_set,

    # --------------------------------------------------------
    # Centrality checks
    # --------------------------------------------------------

    "centralities_finite":
        sna_checks["centralities_finite"],

    "degree_nonnegative":
        sna_checks["degree_nonnegative"],

    "strength_nonnegative":
        sna_checks["strength_nonnegative"],

    # --------------------------------------------------------
    # Community checks
    # --------------------------------------------------------

    "community_assignment_complete":
        sna_checks["community_covers_all_nodes"],

    "community_assignment_unique":
        sna_checks["community_assignment_unique"],

    "community_summary_created":
        len(community_summary) > 0,

    "component_summary_created":
        len(component_table) > 0,

    # --------------------------------------------------------
    # Sensitivity analysis
    # --------------------------------------------------------

    "structural_sensitivity_completed":
        len(community_sensitivity) == 2,

    # --------------------------------------------------------
    # Output files
    # --------------------------------------------------------

    "centrality_output_exists":
        CENTRALITY_OUTPUT.exists(),

    "community_output_exists":
        COMMUNITY_OUTPUT.exists(),

    "community_summary_output_exists":
        COMMUNITY_SUMMARY_OUTPUT.exists(),

    "component_output_exists":
        COMPONENT_OUTPUT.exists(),

    "centrality_summary_output_exists":
        CENTRALITY_SUMMARY_OUTPUT.exists(),

    "community_network_output_exists":
        COMMUNITY_NETWORK_OUTPUT.exists(),

    # --------------------------------------------------------
    # Leakage protection
    # --------------------------------------------------------

    "target_not_used":
        True,
}


print("=" * 75)
print("NOTEBOOK 07 — FINAL CHECKPOINT")
print("=" * 75)

for name, passed in final_checks.items():

    print(
        f"{'PASS' if passed else 'FAIL':<6} | {name}"
    )

print("=" * 75)


# ------------------------------------------------------------
# Save checkpoint
# ------------------------------------------------------------

FINAL_CHECK_PATH.write_text(
    json.dumps(
        {
            "notebook": "07_sna_communities",
            "seed": int(SEED),
            "network_type": network_config["network_type"],
            "similarity_metric": network_config["similarity_metric"],
            "primary_k": int(network_config["primary_k"]),
            "community_method": "louvain",
            "sensitivity_method": "label_propagation",
            "nodes": int(G.number_of_nodes()),
            "edges": int(G.number_of_edges()),
            "density": float(nx.density(G)),
            "connected_components": int(len(components_sorted)),
            "largest_component_fraction": float(
                largest_fraction
            ),
            "weighted_modularity": float(modularity),
            "community_count": int(
                len(communities_louvain)
            ),
            "target_used_for_sna": False,
            "final_checks": {
                str(k): bool(v)
                for k, v in final_checks.items()
            },
        },
        indent=2
    ),
    encoding="utf-8"
)


if all(final_checks.values()):

    print("OVERALL RESULT: PASS")
    print("Proceed to MANUAL REVIEW.")

else:

    print("OVERALL RESULT: FAIL")
    print(
        "Fix the failed checks before moving "
        "to Notebook 08."
    )

NOTEBOOK 07 — FINAL CHECKPOINT
PASS   | network_nodes_match_train
PASS   | network_edges_positive
PASS   | no_self_loops
PASS   | sna_rows_match_train
PASS   | sna_patient_ids_unique
PASS   | all_train_patients_have_sna
PASS   | centralities_finite
PASS   | degree_nonnegative
PASS   | strength_nonnegative
PASS   | community_assignment_complete
PASS   | community_assignment_unique
PASS   | community_summary_created
PASS   | component_summary_created
PASS   | structural_sensitivity_completed
PASS   | centrality_output_exists
PASS   | community_output_exists
PASS   | community_summary_output_exists
PASS   | component_output_exists
PASS   | centrality_summary_output_exists
PASS   | community_network_output_exists
PASS   | target_not_used
OVERALL RESULT: PASS
Proceed to MANUAL REVIEW.


# Manual GO / STOP Review

Before Notebook 08, inspect:

- `07_centrality.csv`
- `07_community_summary.csv`
- `07_component_summary.csv`
- `07_community_network_summary.csv`

### GO

Proceed if:

- the giant component remains dominant;
- communities are not trivially one giant group plus thousands of singletons;
- centrality distributions are plausible;
- community assignments are complete;
- the network structure is interpretable as clinical similarity.

### STOP

Stop if community detection produces clearly pathological structure or if centrality
values are numerically unstable.

### Next

**Notebook 08 — Baseline Readmission Model**

Notebook 08 will test whether the clinical representation and network variables contain
predictive information about the observed readmission target, while keeping validation
and test usage strictly controlled.

In [30]:
# ============================================================
# SAVE FINAL NOTEBOOK 07 SUMMARY
# ============================================================

summary = {
    "notebook": "07_sna_communities",

    "seed": int(SEED),

    # --------------------------------------------------------
    # Frozen network configuration
    # --------------------------------------------------------

    "network_type": network_config["network_type"],
    "similarity_metric": network_config["similarity_metric"],
    "primary_k": int(network_config["primary_k"]),

    # --------------------------------------------------------
    # Network structure
    # --------------------------------------------------------

    "nodes": int(G.number_of_nodes()),
    "edges": int(G.number_of_edges()),
    "density": float(nx.density(G)),

    "connected_components": int(
        len(components_sorted)
    ),

    "largest_component_fraction": float(
        largest_fraction
    ),

    "isolated_nodes": int(
        network_structure["isolated_nodes"]
    ),

    # --------------------------------------------------------
    # Centrality
    # --------------------------------------------------------

    "betweenness_method":
        "approximate_sampled_sources",

    "betweenness_samples":
        200,

    "closeness_method":
        "landmark_based_approximation",

    "closeness_samples":
        100,

    # --------------------------------------------------------
    # Community detection
    # --------------------------------------------------------

    "community_method": "louvain",

    "community_count": int(
        len(communities_louvain)
    ),

    "largest_community_size": int(
        max(
            len(c)
            for c in communities_louvain
        )
    ),

    "weighted_modularity": float(
        modularity
    ),

    # --------------------------------------------------------
    # Sensitivity analysis
    # --------------------------------------------------------

    "sensitivity_method":
        "label_propagation",

    "sensitivity_community_count": int(
        len(labelprop_communities)
    ),

    "sensitivity_largest_community_size": int(
        max(
            len(c)
            for c in labelprop_communities
        )
    ),

    # --------------------------------------------------------
    # Leakage protection
    # --------------------------------------------------------

    "target_used_for_sna": False,

    # --------------------------------------------------------
    # Final checks
    # --------------------------------------------------------

    "final_checks": {
        str(k): bool(v)
        for k, v in final_checks.items()
    }
}


# ============================================================
# SAVE SUMMARY
# ============================================================

SUMMARY_OUTPUT = (
    RESULTS_DIR / "07_sna_summary.json"
)

SUMMARY_OUTPUT.write_text(
    json.dumps(
        summary,
        indent=2
    ),
    encoding="utf-8"
)

print("=" * 70)
print("NOTEBOOK 07 SUMMARY SAVED")
print("=" * 70)
print("File:", SUMMARY_OUTPUT)
print()
print("Community method: Louvain")
print(
    "Communities:",
    len(communities_louvain)
)
print(
    "Weighted modularity:",
    round(float(modularity), 6)
)
print(
    "Sensitivity communities:",
    len(labelprop_communities)
)
print()
print("Summary save: PASS")

NOTEBOOK 07 SUMMARY SAVED
File: C:\Users\Gayatri\OneDrive\Desktop\sna\results\07_sna_summary.json

Community method: Louvain
Communities: 105
Weighted modularity: 0.86524
Sensitivity communities: 1361

Summary save: PASS
